# Building Your First AI Chatbot

Welcome! In this lab, you'll build your own AI chatbot.

## About This Environment

You're running this notebook on an **AMD MI300 GPU** in the **AMD Developer Cloud**. The MI300 is a powerful GPU with 192GB of high-bandwidth memory, making it perfect for running large language models quickly and efficiently.

Throughout this tutorial, when you chat with your AI bot, the inference is running on AMD hardware!

## What You'll Learn

1. [What is inference?](#what-is-inference)
2. [Loading the AI model](#loading-model)
3. [Building a basic chatbot](#basic-chatbot)
4. [Customizing with system prompts](#custom-chatbot)
5. [Controlling AI behavior with parameters](#parameters)
6. [Adding a web interface](#widget-gui)

---

<a id="what-is-inference"></a>

## What is Inference?

If you've chatted with AI bots like ChatGPT or Claude, you've already used **inference**!

**Inference** is when you send an input to a trained generative AI model and it generates an output (e.g. text, code, image).

In this tutorial, we'll use **vLLM** to host and run a model directly on AMD's MI300 GPU.

---

<a id="loading-model"></a>

## Loading the AI Model

Before we can chat, we need to load a pre-trained model onto the MI300 GPU. We'll use **Qwen 3**, a popular open-source language model with over 3 million downloads.

Run the cell below. Note that it may take a minute or two to load the model into GPU memory.

In [ ]:
from vllm import LLM, SamplingParams

# Load the AI model on the MI300 GPU
# This takes about a minute - you'll see loading messages
model_name = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Loading {model_name} on AMD MI300 GPU...")
llm = LLM(model=model_name)
print("✓ Model loaded and ready!")

<a id="basic-chatbot"></a>

## Building a Basic Chatbot

Great! Now that the model is loaded, let's build a simple chatbot.

The code below creates a conversation loop where you can chat with your AI bot. After you run the cell, you can type your messages and press Enter to converse. Give it a try! When you're ready to continue, type `exit` or `quit` to end the chat.

In [ ]:
# Store our conversation history
conversation = []

# Set a reasonable response length
sampling_params = SamplingParams(max_tokens=200)

# Chat loop
while True:
    user_input = input("You: ")
    
    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break
    
    # Add user message to conversation
    conversation.append({"role": "user", "content": user_input})
    
    # This is INFERENCE - send to the model and get a response!
    outputs = llm.chat(conversation, sampling_params)
    bot_message = outputs[0].outputs[0].text
    
    # Add bot's response to conversation so it remembers context
    conversation.append({"role": "assistant", "content": bot_message})
    
    print(f"Bot: {bot_message}\n")

### What Just Happened?

Congratulations, you just performed **inference**!

Here's what happened each time you sent a message:

1. Your text was added to the conversation history
2. The conversation was sent to the Qwen 3 model running on the **MI300 GPU**
3. The GPU processed billions of calculations in parallel
4. The model generated a response based on patterns learned during training
5. The response came back to you in seconds

This is how ChatGPT, Claude, and other AI chatbots work!

**Note:** The chatbot remembers your conversation because we save both your messages and its responses in the `conversation` list.

---

<a id="custom-chatbot"></a>

## Customizing Your Chatbot with System Prompts

Want to give your chatbot a personality? That's where **system prompts** come in!

A **system prompt** is a special instruction that tells the AI how to behave. It's like giving the chatbot a role to play.

Let's create a Shakespeare chatbot that speaks in Elizabethan English. Run the cell and try a few messages! When you're ready to continue, type `exit` or `quit` to end the chat.

In [ ]:
# Create a conversation with a system prompt
shakespeare_conversation = [
    {
        "role": "system",  # This is the special system message
        "content": "You are a chatbot that speaks like William Shakespeare. Use Elizabethan English and poetic language. Be helpful but stay in character!"
    }
]

# Set a reasonable response length
sampling_params = SamplingParams(max_tokens=200)

# Chat loop
while True:
    user_input = input("You: ")
    
    if user_input.lower() in ["exit", "quit"]:
        print("Farewell!")
        break
    
    shakespeare_conversation.append({"role": "user", "content": user_input})
    
    outputs = llm.chat(shakespeare_conversation, sampling_params)
    bot_message = outputs[0].outputs[0].text
    
    shakespeare_conversation.append({"role": "assistant", "content": bot_message})
    
    print(f"Shakespeare Bot: {bot_message}\n")

### How System Prompts Work

Notice how the chatbot's personality completely changed? That's the power of system prompts!

The conversation list starts with a message that has:
- `"role": "system"` - tells the AI this is an instruction, not a user message
- `"content": "..."` - the actual instruction about how to behave

This is how bots are polite, how coding assistants know to write code, and how different AI tools have different personalities.

### Try It Yourself!

Create your own custom chatbot! Some ideas:
- **Pirate Bot**: "You are a friendly pirate. Talk like a pirate!"
- **Rap Bot**: "You are a talented rapper. Respond only in rhymes and rap lyrics."
- **Cat Bot**: "You are a cat. Respond with meows."
- **Science Bot**: "You are a scientist who explains things with facts and examples."

Modify the system prompt below to see how the model responds differently to "Hello! Tell me about yourself".

In [ ]:
# Create your own custom chatbot!
my_conversation = [
    {
        "role": "system",
        "content": "You are a friendly pirate. Talk like a pirate!"  # <-- Change this!
    }
]

# Test it with a quick message
my_conversation.append({"role": "user", "content": "Hello! Tell me about yourself."})

sampling_params = SamplingParams(max_tokens=200)
outputs = llm.chat(my_conversation, sampling_params)

print(outputs[0].outputs[0].text)

<a id="parameters"></a>

## Controlling AI Behavior with Parameters

You've already been using `SamplingParams` with `max_tokens` to control response length. But there are other parameters that control **how** the AI generates text:

### The Three Key Parameters

1. **temperature** - Controls creativity and randomness
2. **max_tokens** - Controls response length (you've been using this!)
3. **top_p** - Controls word choice diversity

Let's explore how temperature affects responses!

In [ ]:
# Let's compare LOW vs HIGH temperature
question = "Tell me a fun fact about cats."

print("🔵 LOW Temperature (0.2) - More predictable and focused:\n")
sampling_params = SamplingParams(temperature=0.2, max_tokens=200)
outputs = llm.chat([{"role": "user", "content": question}], sampling_params)
print(outputs[0].outputs[0].text)

print("\n" + "="*60 + "\n")

print("🔴 HIGH Temperature (0.9) - More creative and varied:\n")
sampling_params = SamplingParams(temperature=0.9, max_tokens=200)
outputs = llm.chat([{"role": "user", "content": question}], sampling_params)
print(outputs[0].outputs[0].text)

print("\n" + "="*60)
print("💡 Notice the difference? Higher temperature = more variety!")

### Parameter Guide

Here's a quick reference for when to use different parameter values:

**temperature** (0.0 to 1.0)
- **0.1-0.3** → Predictable, consistent, factual (good for Q&A, facts)
- **0.5-0.7** → Balanced (good for general chatting)
- **0.8-1.0** → Creative, varied, surprising (good for storytelling, brainstorming)

**max_tokens**
- **50-100** → Short, concise answers
- **150-300** → Medium responses (good default)
- **500+** → Long, detailed explanations

**top_p** (0.0 to 1.0)
- **0.1-0.5** → Conservative word choices
- **0.7-0.9** → Balanced (good default is 0.9)
- **0.95-1.0** → Maximum diversity

Now try experimenting yourself!

In [ ]:
# Experiment with different parameter combinations!
# Try changing these values and see how the response changes

sampling_params = SamplingParams(
    temperature=0.7,    # Try: 0.2 (safe), 0.7 (balanced), or 0.9 (creative)
    max_tokens=200,     # Try: 50 (short), 200 (medium), or 300 (long)
    top_p=0.9          # Try: 0.5 (more focused), 0.9 (balanced), or 1.0 (more diverse)
)

outputs = llm.chat(
    [{"role": "user", "content": "What's the best way to learn AI?"}],
    sampling_params
)

print(outputs[0].outputs[0].text)

<a id="widget-gui"></a>

## Adding a Graphical Interface

Want a real chat interface? We can use **ipywidgets** to create a quick interactive UI with sliders! 

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create widgets
message_box = widgets.Textarea(
    placeholder='Type your message here...',
    description='Message:',
    layout=widgets.Layout(width='100%', height='80px')
)

temperature_slider = widgets.FloatSlider(
    value=0.7,
    min=0.1,
    max=1.0,
    step=0.1,
    description='Temperature:',
    style={'description_width': '150px'}
)

max_tokens_slider = widgets.IntSlider(
    value=200,
    min=50,
    max=500,
    step=10,
    description='Max Tokens:',
    style={'description_width': '150px'}
)

top_p_slider = widgets.FloatSlider(
    value=0.9,
    min=0.1,
    max=1.0,
    step=0.1,
    description='Top P:',
    style={'description_width': '150px'}
)

send_button = widgets.Button(
    description='Send',
    button_style='success',
    icon='paper-plane'
)

output_area = widgets.Output()

# Chat function
def on_send_clicked(b):
    with output_area:
        clear_output()
        message = message_box.value
        if message.strip():
            print(f"You: {message}\n")
            
            sampling_params = SamplingParams(
                temperature=temperature_slider.value,
                max_tokens=int(max_tokens_slider.value),
                top_p=top_p_slider.value
            )
            
            outputs = llm.chat([{"role": "user", "content": message}], sampling_params)
            response = outputs[0].outputs[0].text
            
            print(f"Bot: {response}")

send_button.on_click(on_send_clicked)

# Display the interface
display(widgets.HTML("<h3>AI Chatbot on AMD MI300</h3>"))
display(widgets.HTML("<p>Chat with AI running on AMD hardware!</p>"))
display(message_box)
display(temperature_slider)
display(max_tokens_slider)
display(top_p_slider)
display(send_button)
display(output_area)

---

## Summary: What You've Learned

Congratulations! You've learned the fundamentals of AI inference on AMD hardware. 🎉

### Key Concepts

**1. Inference**
- Inference means using a trained AI model to generate outputs
- Every time you sent a message to the chatbot, you performed inference
- The AMD MI300 GPU made this fast by doing billions of calculations in parallel

**2. System Prompts**  
- System prompts tell the AI how to behave
- They give the chatbot a personality or role
- This is how AI tools can get specific behaviors

**3. Parameters**
- `temperature` → Controls creativity
- `max_tokens` → Controls response length
- `top_p` → Controls word diversity

### The Core Code Pattern

Here's the simple pattern you learned:

```python
# 1. Load model (once)
llm = LLM(model="Qwen/Qwen3-4B-Instruct-2507")

# 2. Create conversation with optional system prompt
conversation = [
    {"role": "system", "content": "Your instructions here"},
    {"role": "user", "content": "Your message"}
]

# 3. Do inference
sampling_params = SamplingParams(temperature=0.7, max_tokens=200)
outputs = llm.chat(conversation, sampling_params)
response = outputs[0].outputs[0].text
```

That's the foundation of generative AI!

### AI on AMD

You experienced real AI running on:
- ✅ **AMD MI300 GPU** - Fast, powerful GPU with 192GB of high-bandwidth memory
- ✅ **vLLM** - Optimized inference engine for AMD ROCm
- ✅ **AMD Developer Cloud** - Cloud access to MI300 GPU

---

### Keep Experimenting!

Try these challenges:
- Make a chatbot that speaks a different language
- Create a chatbot that rhymes every response
- Build a quiz bot that asks questions
- Experiment with extreme temperature values (0.1 vs 1.0)

And see how easy it can be to build from — like making a quick GUI with widgets, adding conversation history, or creating specialized assistants for different tasks!